# Module 04: Python Data & APIs
**Project:** Data Analysis Application (Restaurant Tips dataset + Chennai Weather REST API)

**Objectives covered**
1. Data analysis fundamentals with NumPy and Pandas
2. Data cleaning, manipulation and visualization with Matplotlib/Seaborn
3. Consuming a REST API (HTTP requests and JSON)
4. A complete application: load, clean, analyze, report

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
print("Libraries loaded")

## 2. NumPy basics

In [ ]:
arr = np.array([10, 20, 30, 40, 50])
print("Array:", arr)
print("Mean:", arr.mean(), "| Std:", arr.std(), "| Sum:", arr.sum())
print("Array * 2:", arr * 2)

matrix = np.arange(1, 10).reshape(3, 3)
print("\n3x3 matrix:\n", matrix)
print("Transpose:\n", matrix.T)
print("Random numbers:", np.random.randint(1, 100, 5))

## 3. Load the dataset (Pandas)

In [ ]:
df = sns.load_dataset("tips")   # built-in dataset, no upload needed
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### Make the data messy on purpose (so we can show cleaning)

In [ ]:
rng = np.random.default_rng(42)
dirty = df.copy()

# add missing values
for col in ["total_bill", "tip", "size"]:
    idx = rng.choice(dirty.index, 8, replace=False)
    dirty.loc[idx, col] = np.nan

# add duplicate rows
dirty = pd.concat([dirty, dirty.sample(10, random_state=1)], ignore_index=True)

print("Dirty shape:", dirty.shape)
print("\nMissing values:\n", dirty.isnull().sum())
print("\nDuplicate rows:", dirty.duplicated().sum())

## 4. Data cleaning

In [ ]:
clean = dirty.copy()

# 1. remove duplicates
clean = clean.drop_duplicates()

# 2. fill missing numeric values with median
for col in ["total_bill", "tip", "size"]:
    clean[col] = clean[col].fillna(clean[col].median())

# 3. fix data types
clean["size"] = clean["size"].astype(int)
for col in ["sex", "smoker", "day", "time"]:
    clean[col] = clean[col].astype(str).str.strip().str.title()

# 4. rename columns
clean = clean.rename(columns={"total_bill": "bill_amount", "size": "party_size"})

# 5. new column
clean["tip_percent"] = (clean["tip"] / clean["bill_amount"] * 100).round(2)

print("Clean shape:", clean.shape)
print("Missing values left:", clean.isnull().sum().sum())
print("Duplicates left:", clean.duplicated().sum())
clean.head()

## 5. Analysis and manipulation

In [ ]:
print("Average tip % by day:")
print(clean.groupby("day")["tip_percent"].mean().round(2).sort_values(ascending=False))

print("\nAverage bill by time and smoker:")
print(clean.pivot_table(values="bill_amount", index="time", columns="smoker", aggfunc="mean").round(2))

print("\nTop 5 biggest bills:")
print(clean.sort_values("bill_amount", ascending=False).head())

print("\nParties with bill > 30 and tip% > 15:", len(clean[(clean.bill_amount > 30) & (clean.tip_percent > 15)]))

## 6. Visualization (Matplotlib / Seaborn)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1 Histogram
sns.histplot(clean["bill_amount"], bins=20, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Distribution of Bill Amount")

# 2 Bar chart
sns.barplot(data=clean, x="day", y="tip_percent", ax=axes[0, 1], errorbar=None)
axes[0, 1].set_title("Average Tip % by Day")

# 3 Scatter
sns.scatterplot(data=clean, x="bill_amount", y="tip", hue="time", ax=axes[0, 2])
axes[0, 2].set_title("Bill vs Tip")

# 4 Box plot
sns.boxplot(data=clean, x="day", y="bill_amount", ax=axes[1, 0])
axes[1, 0].set_title("Bill Amount by Day")

# 5 Count plot
sns.countplot(data=clean, x="party_size", ax=axes[1, 1])
axes[1, 1].set_title("Party Size Count")

# 6 Heatmap
sns.heatmap(clean[["bill_amount", "tip", "party_size", "tip_percent"]].corr(),
            annot=True, cmap="coolwarm", ax=axes[1, 2])
axes[1, 2].set_title("Correlation Heatmap")

plt.tight_layout()
plt.savefig("tips_dashboard.png", dpi=120)
plt.show()

## 7. REST API: Open-Meteo weather for Chennai (no API key needed)

In [ ]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 13.0,
    "longitude": 80.2,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "Asia/Kolkata",
    "past_days": 60,
    "forecast_days": 1,
}

response = requests.get(url, params=params, timeout=30)
print("Status code:", response.status_code)
print("URL called :", response.url)

if response.status_code == 200:
    data = response.json()
    print("\nTop-level JSON keys:", list(data.keys()))
    print(json.dumps(data["daily"], indent=2)[:400], "...")
else:
    raise Exception(f"API failed: {response.status_code}")

In [ ]:
# JSON -> DataFrame
weather = pd.DataFrame(data["daily"])
weather = weather.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max",
    "temperature_2m_min": "temp_min",
    "precipitation_sum": "rain_mm",
})
weather["date"] = pd.to_datetime(weather["date"])
weather = weather.dropna().drop_duplicates()
weather["temp_avg"] = ((weather["temp_max"] + weather["temp_min"]) / 2).round(2)
weather["rainy"] = weather["rain_mm"] > 0
weather.head()

In [ ]:
print("Hottest day :", weather.loc[weather.temp_max.idxmax(), ["date", "temp_max"]].tolist())
print("Coolest day :", weather.loc[weather.temp_min.idxmin(), ["date", "temp_min"]].tolist())
print("Rainy days  :", int(weather.rainy.sum()), "of", len(weather))
print("Total rain  :", round(weather.rain_mm.sum(), 1), "mm")

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
ax[0].plot(weather["date"], weather["temp_max"], label="Max", color="tomato")
ax[0].plot(weather["date"], weather["temp_min"], label="Min", color="steelblue")
ax[0].set_title("Chennai Temperature (last 60 days)")
ax[0].set_ylabel("°C"); ax[0].legend(); ax[0].tick_params(axis="x", rotation=45)

ax[1].bar(weather["date"], weather["rain_mm"], color="teal")
ax[1].set_title("Chennai Daily Rainfall")
ax[1].set_ylabel("mm"); ax[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("weather_charts.png", dpi=120)
plt.show()

## 8. Final Data Analysis Application (load, clean, analyze, report)

In [ ]:
from datetime import datetime

def run_analysis_app(dataset_name="tips", lat=13.0, lon=80.2, city="Chennai"):
    lines = []
    lines.append("=" * 55)
    lines.append("DATA ANALYSIS REPORT")
    lines.append("Generated: " + datetime.now().strftime("%d-%m-%Y %H:%M"))
    lines.append("=" * 55)

    # ---- Part 1: dataset ----
    raw = sns.load_dataset(dataset_name)
    raw = raw.drop_duplicates().dropna()
    raw["tip_percent"] = raw["tip"] / raw["total_bill"] * 100
    lines.append(f"\n[1] DATASET: {dataset_name}  ({raw.shape[0]} rows, {raw.shape[1]} cols)")
    lines.append(f"    Average bill     : {raw.total_bill.mean():.2f}")
    lines.append(f"    Average tip      : {raw.tip.mean():.2f}")
    lines.append(f"    Average tip %    : {raw.tip_percent.mean():.2f}%")
    best_day = raw.groupby("day", observed=True)["tip_percent"].mean().idxmax()
    lines.append(f"    Best tipping day : {best_day}")
    lines.append(f"    Bill/Tip correlation: {raw.total_bill.corr(raw.tip):.2f}")

    # ---- Part 2: API ----
    try:
        r = requests.get("https://api.open-meteo.com/v1/forecast", params={
            "latitude": lat, "longitude": lon,
            "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
            "timezone": "Asia/Kolkata", "past_days": 60, "forecast_days": 1}, timeout=30)
        r.raise_for_status()
        w = pd.DataFrame(r.json()["daily"]).dropna()
        lines.append(f"\n[2] WEATHER API: {city}  ({len(w)} days)")
        lines.append(f"    Avg max temp : {w.temperature_2m_max.mean():.1f} C")
        lines.append(f"    Avg min temp : {w.temperature_2m_min.mean():.1f} C")
        lines.append(f"    Rainy days   : {(w.precipitation_sum > 0).sum()}")
        lines.append(f"    Total rain   : {w.precipitation_sum.sum():.1f} mm")
        w.to_csv("weather_data.csv", index=False)
    except Exception as e:
        lines.append(f"\n[2] WEATHER API failed: {e}")

    raw.to_csv("cleaned_dataset.csv", index=False)
    report = "\n".join(lines)
    with open("analysis_report.txt", "w", encoding="utf-8") as f:
        f.write(report)
    return report

print(run_analysis_app())

In [ ]:
# Optional: download generated files from Colab
# from google.colab import files
# for f in ["analysis_report.txt", "cleaned_dataset.csv", "weather_data.csv", "tips_dashboard.png", "weather_charts.png"]:
#     files.download(f)

## 9. Conclusion
- Used **NumPy** for arrays and statistics, and **Pandas** for loading, cleaning (duplicates, missing values, types, renaming) and grouping.
- Built **6 charts** with Matplotlib/Seaborn: histogram, bar, scatter, box, count and heatmap, plus line and bar charts for the weather.
- Consumed a **REST API** with `requests`, checked the status code, parsed the **JSON** and converted it to a DataFrame.
- Combined everything into one **application** that generates a text report and CSV outputs.